# Cena: IA de vídeo realista, 100% gratuita

Modelo **Wan** (código aberto, Apache 2.0) rodando em GPU gratuita.

**Google Colab (só precisa de uma conta Google, sem telefone):**
1. Menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4 → Salvar**.
2. Menu **Ambiente de execução → Executar tudo**.
3. Se aparecer um aviso de que o notebook não é do Google, clique em **Executar mesmo assim**.

A Cena detecta a pouca RAM do Colab grátis e entra sozinha no **modo econômico**.

**Kaggle** (exige verificação de telefone): *Accelerator* **GPU T4 x2**, *Internet* **On**, *Run all*.

A primeira execução baixa ~20–30 GB de pesos do modelo, o que leva alguns minutos.
Na T4 gratuita, cada vídeo leva **de ~5 a ~30 min**, conforme duração, resolução e qualidade.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
# No Kaggle, /kaggle/working tem só 20 GB: o cache do modelo vai para /tmp, que é maior.
os.environ.setdefault("HF_HOME", "/tmp/hf" if os.path.exists("/kaggle") else os.path.expanduser("~/.cache/huggingface"))

In [ ]:
%pip install -q -U diffusers transformers accelerate ftfy sentencepiece imageio imageio-ffmpeg gradio

In [ ]:
!mkdir -p cena

In [ ]:
%%writefile cena/__init__.py
"""Cena: IA de vídeo realista, gratuita e de código aberto, sobre o modelo Wan (Apache 2.0)."""

__version__ = "0.1.0"

In [ ]:
%%writefile cena/director.py
"""O "diretor": transforma um pedido simples num prompt de cinema realista.

Modelos de vídeo respondem muito melhor a linguagem de câmera (lente, luz,
movimento) do que a descrições soltas. Aqui ficam os presets de estilo e o
prompt negativo que afasta o "visual de IA" (pele de plástico, mãos erradas,
cara de 3D).
"""

from dataclasses import dataclass

REALISMO = (
    "photorealistic live-action footage, real people, natural skin texture with visible pores, "
    "anatomically correct hands, physically accurate lighting and shadows, realistic motion, "
    "shallow depth of field, subtle film grain, natural color grading"
)

ESTILOS: dict[str, str] = {
    "cinema": "cinematic shot on a cinema camera, 35mm anamorphic lens, soft key light, gentle camera movement",
    "comercial": "premium commercial, clean composition, soft natural window light, slow dolly-in, polished but natural",
    "documentario": "documentary style, handheld camera, available light, candid and unposed moment",
    "celular": "shot on a smartphone, vertical-friendly framing, handheld, everyday real life, natural daylight",
    "noturno": "night scene, practical lights, neon and street lamps, high dynamic range, cinematic low light",
    "drone": "aerial drone shot, smooth forward motion, golden hour, wide landscape, realistic atmosphere",
}

MOVIMENTOS: dict[str, str] = {
    "auto": "",
    "parada": "static locked-off camera",
    "aproximando": "slow push-in towards the subject",
    "afastando": "slow pull-back revealing the scene",
    "acompanhando": "tracking shot following the subject",
    "orbitando": "slow orbit around the subject",
    "panoramica": "slow pan from left to right",
}

# Base: o prompt negativo recomendado pelos autores do Wan, mais o que denuncia vídeo de IA.
NEGATIVO = (
    "bright tones, overexposed, static, blurred details, subtitles, style, works, paintings, images, "
    "overall gray, worst quality, low quality, JPEG compression residue, ugly, incomplete, extra fingers, "
    "poorly drawn hands, poorly drawn faces, deformed, disfigured, misshapen limbs, fused fingers, "
    "still picture, messy background, three legs, many people in the background, walking backwards, "
    "cartoon, anime, 3d render, cgi, illustration, plastic skin, waxy skin, doll-like face, "
    "text, watermark, logo, morphing, flickering, distorted face"
)


@dataclass(frozen=True)
class Direcao:
    prompt: str
    negativo: str


def dirigir(pedido: str, estilo: str = "cinema", movimento: str = "auto", extra_negativo: str = "") -> Direcao:
    """Monta o prompt final. O pedido do usuário vem primeiro: é o que mais pesa para o modelo."""
    pedido = " ".join(pedido.split())
    if not pedido:
        raise ValueError("Descreva a cena que você quer ver.")
    if estilo not in ESTILOS:
        raise ValueError(f"Estilo desconhecido: {estilo}. Opções: {', '.join(ESTILOS)}")
    if movimento not in MOVIMENTOS:
        raise ValueError(f"Movimento desconhecido: {movimento}. Opções: {', '.join(MOVIMENTOS)}")
    partes = [pedido.rstrip("."), ESTILOS[estilo], MOVIMENTOS[movimento], REALISMO]
    negativo = NEGATIVO + (f", {extra_negativo.strip()}" if extra_negativo.strip() else "")
    return Direcao(prompt=". ".join(p for p in partes if p) + ".", negativo=negativo)


def quadros_para(segundos: float, fps: int) -> int:
    """O Wan exige número de quadros no formato 4k+1 (ex.: 49, 81, 121)."""
    k = max(1, round(segundos * fps / 4))
    return 4 * k + 1

In [ ]:
%%writefile cena/engine.py
"""Motor de geração: carrega o modelo Wan e transforma uma Direcao em MP4.

Dois modos, escolhidos pelo que cabe na GPU (gratuita) disponível:
  - "maxima": Wan 2.2 TI2V-5B, até 1280×704 a 24 fps (melhor realismo).
  - "rapida": Wan 2.1 T2V-1.3B, 832×480 a 16 fps (cabe em ~8 GB de VRAM).
Tudo de código aberto e gratuito; os pesos são baixados do Hugging Face na
primeira execução.
"""

from __future__ import annotations

import gc
import os
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable

from .director import Direcao, quadros_para


@dataclass(frozen=True)
class Modelo:
    nome: str
    repo: str
    fps: int
    resolucoes: dict[str, tuple[int, int]]  # rótulo -> (largura, altura)
    passos: int
    guidance: float
    flow_shift: float
    vram_ideal_gb: float  # abaixo disso, usa descarregamento para a RAM


MODELOS: dict[str, Modelo] = {
    "maxima": Modelo(
        nome="Máxima (Wan 2.2 · 5B)",
        repo=os.environ.get("CENA_MODELO_MAXIMA", "Wan-AI/Wan2.2-TI2V-5B-Diffusers"),
        fps=24,
        resolucoes={"720p": (1280, 704), "480p": (832, 480)},
        passos=50,
        guidance=5.0,
        flow_shift=5.0,
        vram_ideal_gb=32,
    ),
    "rapida": Modelo(
        nome="Rápida (Wan 2.1 · 1.3B)",
        repo=os.environ.get("CENA_MODELO_RAPIDA", "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"),
        fps=16,
        resolucoes={"480p": (832, 480)},
        passos=40,
        guidance=5.0,
        flow_shift=3.0,
        vram_ideal_gb=16,
    ),
}


@dataclass
class Pedido:
    direcao: Direcao
    modo: str = "maxima"
    resolucao: str = "720p"
    segundos: float = 5.0
    semente: int = -1
    passos: int | None = None
    guidance: float | None = None
    vertical: bool = False


@dataclass
class Resultado:
    arquivo: Path
    semente: int
    segundos_gerando: float
    largura: int
    altura: int
    quadros: int
    fps: int


Progresso = Callable[[int, int], None]


def ram_gb() -> float:
    try:
        return os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
    except (ValueError, OSError, AttributeError):
        return 0.0


def usar_economia() -> bool:
    """Pouca RAM (Colab grátis tem ~12 GB): nunca deixa o modelo inteiro na memória do sistema."""
    forcado = os.environ.get("CENA_ECONOMIA")
    if forcado is not None:
        return forcado == "1"
    return 0 < ram_gb() < 24


def info_gpu() -> dict[str, Any]:
    """Descreve a GPU (ou a falta dela) para escolher modo e descarregamento."""
    try:
        import torch
    except ImportError:
        return {"cuda": False, "nome": "sem PyTorch", "vram_gb": 0.0}
    if not torch.cuda.is_available():
        return {"cuda": False, "nome": "sem GPU", "vram_gb": 0.0}
    p = torch.cuda.get_device_properties(0)
    return {"cuda": True, "nome": p.name, "vram_gb": p.total_memory / 1024**3}


def modo_recomendado(vram_gb: float) -> str:
    # O 5B roda com descarregamento a partir de ~15 GB (T4 do Kaggle/Colab); abaixo disso, o 1.3B.
    return "maxima" if vram_gb >= 14.5 else "rapida"


class Motor:
    """Mantém um pipeline carregado por vez (troca de modo libera a VRAM do anterior)."""

    def __init__(self, saida: Path | str = "videos", carregar: Callable[[Modelo], Any] | None = None):
        self.saida = Path(saida)
        self.saida.mkdir(parents=True, exist_ok=True)
        self._carregar = carregar or carregar_pipeline
        self._modo: str | None = None
        self._pipe: Any = None

    def pipeline(self, modo: str) -> Any:
        if modo not in MODELOS:
            raise ValueError(f"Modo desconhecido: {modo}")
        if self._modo != modo:
            self.descarregar()
            self._pipe = self._carregar(MODELOS[modo])
            self._modo = modo
        return self._pipe

    def descarregar(self) -> None:
        self._pipe = None
        self._modo = None
        gc.collect()
        try:
            import torch

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except ImportError:
            pass

    def gerar(self, pedido: Pedido, progresso: Progresso | None = None) -> Resultado:
        m = MODELOS[pedido.modo]
        resolucao = pedido.resolucao if pedido.resolucao in m.resolucoes else next(iter(m.resolucoes))
        largura, altura = m.resolucoes[resolucao]
        if pedido.vertical:
            largura, altura = altura, largura
        quadros = quadros_para(pedido.segundos, m.fps)
        passos = pedido.passos or m.passos
        semente = pedido.semente if pedido.semente >= 0 else int.from_bytes(os.urandom(4), "big") % 2**31

        pipe = self.pipeline(pedido.modo)
        inicio = time.time()
        quadros_gerados = self._executar(
            pipe,
            prompt=pedido.direcao.prompt,
            negative_prompt=pedido.direcao.negativo,
            width=largura,
            height=altura,
            num_frames=quadros,
            num_inference_steps=passos,
            guidance_scale=pedido.guidance or m.guidance,
            semente=semente,
            progresso=progresso,
            total=passos,
        )
        arquivo = self.saida / f"cena-{time.strftime('%Y%m%d-%H%M%S')}-{semente}.mp4"
        exportar_mp4(quadros_gerados, arquivo, m.fps)
        return Resultado(arquivo, semente, time.time() - inicio, largura, altura, quadros, m.fps)

    def _executar(self, pipe: Any, *, semente: int, progresso: Progresso | None, total: int, **kwargs: Any) -> Any:
        import torch

        gerador = torch.Generator(device="cpu").manual_seed(semente)

        def ao_fim_do_passo(_pipe: Any, passo: int, _t: Any, cb_kwargs: dict) -> dict:
            if progresso:
                progresso(passo + 1, total)
            return cb_kwargs

        saida = pipe(generator=gerador, callback_on_step_end=ao_fim_do_passo, **kwargs)
        return saida.frames[0]


def _dtype() -> Any:
    import torch

    # bf16 é o formato nativo do Wan, mas só GPUs Ampere+ (RTX 30xx, A100...) o fazem em hardware.
    # A T4 gratuita do Kaggle/Colab emularia bf16 muitas vezes mais devagar, então nela usamos fp16.
    # CENA_DTYPE=bf16|fp16 força a escolha.
    escolha = os.environ.get("CENA_DTYPE")
    nativo_bf16 = torch.cuda.get_device_capability(0)[0] >= 8
    return torch.bfloat16 if escolha == "bf16" or (escolha is None and nativo_bf16) else torch.float16


def _exigir_gpu() -> None:
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError(
            "Nenhuma GPU encontrada. No Kaggle: Settings → Accelerator → GPU T4 x2. "
            "No Colab: Ambiente de execução → Alterar tipo → GPU T4."
        )


def carregar_pipeline(m: Modelo) -> Any:
    """Carrega o Wan com as otimizações que fazem caber numa GPU gratuita."""
    import torch
    from diffusers import AutoencoderKLWan, WanPipeline
    from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler

    _exigir_gpu()
    if usar_economia():
        return PipelineEconomico(m)
    dtype = _dtype()
    vae = AutoencoderKLWan.from_pretrained(m.repo, subfolder="vae", torch_dtype=torch.float32)
    pipe = WanPipeline.from_pretrained(m.repo, vae=vae, torch_dtype=dtype)
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=m.flow_shift)

    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    if vram >= m.vram_ideal_gb:
        pipe.to("cuda")
    else:
        # Mantém só o componente em uso na GPU (codificador de texto → transformer → VAE).
        pipe.enable_model_cpu_offload()
    # Decodifica o vídeo em blocos: evita estourar a memória no último passo.
    pipe.vae.enable_tiling()

    lora = os.environ.get("CENA_LORA")
    if lora:
        # Opcional: LoRA de personagem/estilo (ou de aceleração) compatível com o modelo carregado.
        pipe.load_lora_weights(lora)
    return pipe


def exportar_mp4(quadros: Any, arquivo: Path, fps: int) -> Path:
    """Grava os quadros (lista de arrays HxWx3 em 0..1 ou 0..255) como MP4 H.264 de alta qualidade."""
    import imageio.v2 as imageio
    import numpy as np

    with imageio.get_writer(str(arquivo), fps=fps, codec="libx264", quality=9, pixelformat="yuv420p", macro_block_size=16) as w:
        for q in quadros:
            a = np.asarray(q)
            if a.dtype != np.uint8:
                a = (np.clip(a, 0, 1) * 255).round().astype(np.uint8)
            w.append_data(a)
    return arquivo


class PipelineEconomico:
    """Wan para máquinas com pouca RAM (ex.: Colab grátis, ~12 GB).

    O codificador de texto (UMT5-XXL, ~11 GB) e o gerador de vídeo não cabem
    juntos na RAM nem na VRAM de uma T4. Então, a cada vídeo:
      1. carrega só o codificador de texto direto na GPU, processa o pedido e o descarta;
      2. carrega (ou reaproveita) o gerador de vídeo direto na GPU, sem passar pela RAM,
         e gera a partir do texto já processado.
    É mais lento por vídeo (recarrega o codificador), mas cabe.
    """

    def __init__(self, m: Modelo):
        self.m = m
        self._pipe: Any = None

    def _codificar(self, prompt: str, negativo: str) -> tuple[Any, Any]:
        import torch
        from diffusers import WanPipeline
        from transformers import AutoTokenizer, UMT5EncoderModel

        dtype = _dtype()
        tokenizer = AutoTokenizer.from_pretrained(self.m.repo, subfolder="tokenizer")
        encoder = UMT5EncoderModel.from_pretrained(self.m.repo, subfolder="text_encoder", dtype=dtype, device_map="cuda")
        # Reaproveita o encode_prompt oficial do pipeline (mesmo pré-processamento e preenchimento).
        so_texto = WanPipeline(tokenizer=tokenizer, text_encoder=encoder, vae=None, scheduler=None, transformer=None)
        with torch.no_grad():
            pos, neg = so_texto.encode_prompt(
                prompt=prompt,
                negative_prompt=negativo,
                do_classifier_free_guidance=True,
                max_sequence_length=512,
                device=torch.device("cuda"),
                dtype=dtype,
            )
        del so_texto, encoder
        gc.collect()
        torch.cuda.empty_cache()
        return pos, neg

    def _gerador(self) -> Any:
        if self._pipe is not None:
            return self._pipe
        import torch
        from diffusers import AutoencoderKLWan, WanPipeline, WanTransformer3DModel
        from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler

        dtype = _dtype()
        transformer = WanTransformer3DModel.from_pretrained(self.m.repo, subfolder="transformer", torch_dtype=dtype, device_map="cuda")
        vae = AutoencoderKLWan.from_pretrained(self.m.repo, subfolder="vae", torch_dtype=torch.float32, device_map="cuda")
        pipe = WanPipeline.from_pretrained(self.m.repo, transformer=transformer, vae=vae, text_encoder=None, tokenizer=None, torch_dtype=dtype)
        pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=self.m.flow_shift)
        pipe.vae.enable_tiling()
        lora = os.environ.get("CENA_LORA")
        if lora:
            pipe.load_lora_weights(lora)
        self._pipe = pipe
        return pipe

    def _liberar_gerador(self) -> None:
        import torch

        self._pipe = None
        gc.collect()
        torch.cuda.empty_cache()

    def __call__(self, *, prompt: str, negative_prompt: str, **kwargs: Any) -> Any:
        import torch

        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        # O 5B (~10 GB) e o codificador de texto (~11 GB) não cabem juntos numa GPU de 15 GB.
        if self.m.vram_ideal_gb > 16 and vram < 26:
            self._liberar_gerador()
        pos, neg = self._codificar(prompt, negative_prompt)
        return self._gerador()(prompt_embeds=pos, negative_prompt_embeds=neg, **kwargs)

In [ ]:
%%writefile cena/app.py
"""Interface web da Cena (Gradio). Roda no Kaggle/Colab gratuito ou no seu PC.

    python -m cena.app            # local: http://127.0.0.1:7860
    python -m cena.app --share    # link público gratuito (útil no Kaggle/Colab)
"""

from __future__ import annotations

import argparse
import threading

from .director import ESTILOS, MOVIMENTOS, dirigir
from .engine import MODELOS, Motor, Pedido, info_gpu, modo_recomendado, usar_economia

EXEMPLOS = [
    ["Um homem de 30 anos acorda com o despertador do celular, se espreguiça e sorri enquanto o sol entra pela janela", "comercial", "aproximando"],
    ["Uma senhora rindo enquanto prepara café numa cozinha antiga do interior de Minas, vapor subindo da chaleira", "documentario", "parada"],
    ["Motociclista atravessando a Avenida Paulista à noite na chuva, reflexos de neon no asfalto molhado", "noturno", "acompanhando"],
    ["Ondas quebrando em falésias ao pôr do sol no litoral nordestino", "drone", "auto"],
]

ESTILO_ROTULOS = {"cinema": "Cinema", "comercial": "Comercial", "documentario": "Documentário", "celular": "Filmado no celular", "noturno": "Noturno", "drone": "Drone"}
MOV_ROTULOS = {"auto": "Automático", "parada": "Câmera parada", "aproximando": "Aproximando", "afastando": "Afastando", "acompanhando": "Acompanhando", "orbitando": "Orbitando", "panoramica": "Panorâmica"}


def criar_app(motor: Motor | None = None):
    import gradio as gr

    motor = motor or Motor()
    trava = threading.Lock()  # uma geração por vez: a GPU gratuita não aguenta duas
    gpu = info_gpu()
    # Com pouca RAM (Colab grátis) a Máxima recarrega o modelo a cada vídeo: começa pela Rápida.
    modo_padrao = modo_recomendado(gpu["vram_gb"]) if gpu["cuda"] and not usar_economia() else "rapida"
    res_padrao = "720p" if gpu["vram_gb"] >= 20 else "480p"

    def gerar(pedido, estilo, movimento, modo, resolucao, segundos, vertical, semente, passos, evitar, progress=gr.Progress()):
        direcao = dirigir(pedido, estilo, movimento, evitar or "")
        progress(0, desc="Carregando o modelo (na primeira vez baixa ~20 GB; demora)…")
        with trava:
            r = motor.gerar(
                Pedido(direcao, modo=modo, resolucao=resolucao, segundos=float(segundos), semente=int(semente), passos=int(passos) or None, vertical=bool(vertical)),
                progresso=lambda p, total: progress(p / total, desc=f"Gerando… passo {p}/{total}"),
            )
        minutos = r.segundos_gerando / 60
        info = (
            f"**Pronto em {minutos:.1f} min** · {r.largura}×{r.altura} · {r.quadros} quadros a {r.fps} fps · semente `{r.semente}`  \n"
            f"Gostou do movimento mas quer variar detalhes? Reuse a semente e mude o texto.\n\n"
            f"<details><summary>Prompt enviado ao modelo</summary>\n\n{direcao.prompt}\n\n</details>"
        )
        return str(r.arquivo), info

    with gr.Blocks(title="Cena — IA de vídeo realista") as demo:
        gr.Markdown(
            "# Cena\n**IA de vídeo realista, 100% gratuita e de código aberto** (modelo Wan, Apache 2.0).  \n"
            f"GPU: **{gpu['nome']}** ({gpu['vram_gb']:.0f} GB). Em GPU gratuita cada vídeo leva de alguns minutos a meia hora — vale a pena esperar."
        )
        with gr.Row():
            with gr.Column(scale=3):
                pedido = gr.Textbox(label="O que deve acontecer na cena?", lines=4, placeholder="Quem, onde, fazendo o quê, com que luz… Quanto mais concreto, mais real.")
                with gr.Row():
                    estilo = gr.Dropdown([(ESTILO_ROTULOS[k], k) for k in ESTILOS], value="cinema", label="Estilo")
                    movimento = gr.Dropdown([(MOV_ROTULOS[k], k) for k in MOVIMENTOS], value="auto", label="Câmera")
                with gr.Row():
                    modo = gr.Dropdown([(m.nome, k) for k, m in MODELOS.items()], value=modo_padrao, label="Qualidade")
                    resolucao = gr.Dropdown(["720p", "480p"], value=res_padrao, label="Resolução")
                    segundos = gr.Slider(2, 5, value=4, step=1, label="Duração (s)")
                    vertical = gr.Checkbox(label="Vertical (Reels/TikTok)", value=False)
                with gr.Accordion("Avançado", open=False):
                    semente = gr.Number(value=-1, precision=0, label="Semente (-1 = aleatória)")
                    passos = gr.Slider(0, 60, value=0, step=1, label="Passos (0 = padrão do modelo; menos = mais rápido, menos detalhe)")
                    evitar = gr.Textbox(label="Evitar também (opcional)", placeholder="ex.: chapéu, multidão")
                botao = gr.Button("Gerar vídeo", variant="primary")
                gr.Examples(EXEMPLOS, inputs=[pedido, estilo, movimento])
            with gr.Column(scale=4):
                video = gr.Video(label="Resultado", autoplay=True)
                info = gr.Markdown()
        botao.click(gerar, [pedido, estilo, movimento, modo, resolucao, segundos, vertical, semente, passos, evitar], [video, info])
    demo.queue(max_size=8)
    return demo


def main() -> None:
    ap = argparse.ArgumentParser(description="Cena — IA de vídeo realista")
    ap.add_argument("--share", action="store_true", help="cria um link público gratuito (Kaggle/Colab)")
    ap.add_argument("--porta", type=int, default=7860)
    ap.add_argument("--saida", default="videos")
    a = ap.parse_args()
    criar_app(Motor(a.saida)).launch(share=a.share, server_name="0.0.0.0", server_port=a.porta)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile cena/cli.py
"""Gera um vídeo direto pelo terminal, sem interface.

    python -m cena.cli "um cachorro correndo na praia ao amanhecer" --estilo drone --segundos 4
"""

import argparse

from .director import ESTILOS, MOVIMENTOS, dirigir
from .engine import MODELOS, Motor, Pedido, info_gpu, modo_recomendado


def main() -> None:
    ap = argparse.ArgumentParser(description="Cena — gera um vídeo realista a partir de um texto")
    ap.add_argument("pedido")
    ap.add_argument("--estilo", choices=list(ESTILOS), default="cinema")
    ap.add_argument("--movimento", choices=list(MOVIMENTOS), default="auto")
    ap.add_argument("--modo", choices=list(MODELOS), default=None)
    ap.add_argument("--resolucao", default="480p")
    ap.add_argument("--segundos", type=float, default=4)
    ap.add_argument("--semente", type=int, default=-1)
    ap.add_argument("--vertical", action="store_true")
    ap.add_argument("--saida", default="videos")
    a = ap.parse_args()

    gpu = info_gpu()
    modo = a.modo or modo_recomendado(gpu["vram_gb"])
    print(f"GPU: {gpu['nome']} ({gpu['vram_gb']:.0f} GB) · modo: {MODELOS[modo].nome}")
    r = Motor(a.saida).gerar(
        Pedido(dirigir(a.pedido, a.estilo, a.movimento), modo=modo, resolucao=a.resolucao, segundos=a.segundos, semente=a.semente, vertical=a.vertical),
        progresso=lambda p, t: print(f"\rpasso {p}/{t}", end="", flush=True),
    )
    print(f"\n{r.arquivo}  ({r.segundos_gerando / 60:.1f} min, semente {r.semente})")


if __name__ == "__main__":
    main()

## Opção 1: interface completa (recomendado)
Rode a célula abaixo e abra o link **`https://….gradio.live`** que aparecer. Ele funciona até no celular.
Deixe a célula rodando enquanto usa; o link vale por 72 h.

In [ ]:
from cena.app import criar_app
criar_app().launch(share=True, debug=True)

## Opção 2: gerar direto aqui no notebook
Pare a célula de cima antes, para liberar a GPU.

In [ ]:
from IPython.display import Video
from cena.director import dirigir
from cena.engine import Motor, Pedido, info_gpu, modo_recomendado

pedido = "Um homem de 30 anos acorda com o despertador do celular, se espreguiça e sorri enquanto o sol entra pela janela"
gpu = info_gpu()
motor = Motor("videos")
r = motor.gerar(
    Pedido(dirigir(pedido, "comercial", "aproximando"), modo=modo_recomendado(gpu["vram_gb"]), resolucao="480p", segundos=4),
    progresso=lambda p, t: print(f"\rpasso {p}/{t}", end=""),
)
print(f"\n{r.arquivo} · {r.segundos_gerando/60:.1f} min · semente {r.semente}")
Video(str(r.arquivo), embed=True, width=832)